# 04. Strategy Backtest Comparison — Finding the Best Technical Analysis Strategy

This notebook backtests **16 technical analysis trading strategies** across multiple Binance USDT pairs and compares their performance metrics:
- **Win Rate** — % of trades that were profitable
- **Average Return per Trade** — Mean % gain/loss per signal
- **Total Cumulative Return** — Compounded return across all signals
- **Max Drawdown** — Worst peak-to-trough decline
- **Profit Factor** — Gross profit / Gross loss
- **Sharpe Ratio** — Risk-adjusted return

### Strategies Tested:
1. RSI Oversold Bounce (RSI < 30)
2. RSI Overbought Short (RSI > 70)
3. Bullish MACD Crossover
4. Bearish MACD Crossover
5. EMA 9/21 Golden Cross
6. EMA 9/21 Death Cross
7. Bollinger Band Bounce (Close < Lower Band)
8. Bollinger Band Rejection (Close > Upper Band)
9. Volume Surge + RSI Confirm (RVOL > 2 & RSI < 45)
10. Triple EMA Uptrend Stack (9 > 21 > 50)
11. Mean Reversion (RSI < 35 & Close < BB Lower)
12. Momentum Breakout (RSI > 55 & MACD Bullish Cross & RVOL > 1.5)13. StochRSI Oversold Cross
14. StochRSI Overbought Cross
15. BB Squeeze Breakout (Keltner Channels)
16. Trend Pullback (EMA 200/50)


In [1]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from backend.binance_client import binance_client
from backend.indicators import enrich_klines_dataframe

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
print('All modules loaded successfully.')

All modules loaded successfully.


## 1. Fetch Historical Data for Multiple Pairs
We'll pull 500 hourly candles (~20 days) for several high-volume USDT pairs to get a robust dataset.

In [2]:
SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'BNBUSDT', 'XRPUSDT', 'ADAUSDT', 'DOGEUSDT', 'AVAXUSDT']
INTERVAL = '1h'
LIMIT = 500
HOLD_PERIODS = [6, 12, 24]  # Hours to hold each trade

# Fetch and enrich data for all symbols
datasets = {}
for sym in SYMBOLS:
    try:
        raw = binance_client.get_klines(sym, interval=INTERVAL, limit=LIMIT)
        df = enrich_klines_dataframe(raw)
        if not df.empty and len(df) >= 100:
            datasets[sym] = df
            print(f'{sym}: {len(df)} candles loaded')
        else:
            print(f'{sym}: Insufficient data, skipped')
    except Exception as e:
        print(f'{sym}: Error - {e}')

print(f'\nTotal pairs loaded: {len(datasets)}')

BTCUSDT: 500 candles loaded
ETHUSDT: 500 candles loaded
SOLUSDT: 500 candles loaded
BNBUSDT: 500 candles loaded
XRPUSDT: 500 candles loaded
ADAUSDT: 500 candles loaded
DOGEUSDT: 500 candles loaded
AVAXUSDT: 500 candles loaded

Total pairs loaded: 8


## 2. Define All 12 Trading Strategies

Each strategy is a function that takes a DataFrame and returns a boolean Series marking entry signals.

In [3]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np

def strat_rsi_oversold(df):
    """Buy when RSI drops below 30"""
    return (df['rsi'] < 30) & (df['rsi'].shift(1) >= 30)

def strat_rsi_overbought(df):
    """Short when RSI rises above 70"""
    return (df['rsi'] > 70) & (df['rsi'].shift(1) <= 70)

def strat_macd_bull_cross(df):
    """Buy on MACD histogram flip positive"""
    return (df['macd_hist'].shift(1) <= 0) & (df['macd_hist'] > 0)

def strat_macd_bear_cross(df):
    """Sell on MACD histogram flip negative"""
    return (df['macd_hist'].shift(1) >= 0) & (df['macd_hist'] < 0)

def strat_ema_golden_cross(df):
    """Buy when EMA 9 crosses above EMA 21"""
    return (df['ema_9'].shift(1) <= df['ema_21'].shift(1)) & (df['ema_9'] > df['ema_21'])

def strat_ema_death_cross(df):
    """Sell when EMA 9 crosses below EMA 21"""
    return (df['ema_9'].shift(1) >= df['ema_21'].shift(1)) & (df['ema_9'] < df['ema_21'])

def strat_bb_lower_bounce(df):
    """Buy when close dips below lower Bollinger Band"""
    return (df['close'] < df['bb_lower']) & (df['close'].shift(1) >= df['bb_lower'].shift(1))

def strat_bb_upper_reject(df):
    """Sell when close exceeds upper Bollinger Band"""
    return (df['close'] > df['bb_upper']) & (df['close'].shift(1) <= df['bb_upper'].shift(1))

def strat_volume_rsi_confirm(df):
    """Buy when RVOL > 2x AND RSI < 45"""
    return (df['rvol'] > 2.0) & (df['rsi'] < 45)

def strat_triple_ema_stack(df):
    """Buy when EMA 9 > EMA 21 > EMA 50 alignment first forms"""
    aligned = (df['ema_9'] > df['ema_21']) & (df['ema_21'] > df['ema_50'])
    prev = (df['ema_9'].shift(1) > df['ema_21'].shift(1)) & (df['ema_21'].shift(1) > df['ema_50'].shift(1))
    return aligned & (~prev)

def strat_mean_reversion(df):
    """Buy when RSI < 35 AND close below lower BB"""
    return (df['rsi'] < 35) & (df['close'] < df['bb_lower'])

def strat_momentum_breakout(df):
    """Buy on MACD bull cross + RSI > 55 + RVOL > 1.5"""
    macd_bull = (df['macd_hist'].shift(1) <= 0) & (df['macd_hist'] > 0)
    return macd_bull & (df['rsi'] > 55) & (df['rvol'] > 1.5)

def strat_stoch_rsi_oversold(df):
    """Buy when StochRSI %K crosses above %D in oversold territory (<20)"""
    return (df['stoch_k'].shift(1) <= df['stoch_d'].shift(1)) & (df['stoch_k'] > df['stoch_d']) & (df['stoch_k'] < 20)

def strat_stoch_rsi_overbought(df):
    """Sell when StochRSI %K crosses below %D in overbought territory (>80)"""
    return (df['stoch_k'].shift(1) >= df['stoch_d'].shift(1)) & (df['stoch_k'] < df['stoch_d']) & (df['stoch_k'] > 80)

def strat_bb_squeeze_breakout(df):
    """Buy when BB goes inside Keltner Channels (Squeeze) and breaks out of upper BB"""
    squeeze_on = (df['bb_upper'] < df['kc_upper']) & (df['bb_lower'] > df['kc_lower'])
    breakout = (df['close'] > df['bb_upper'])
    return squeeze_on & breakout

def strat_ema_pullback(df):
    """Buy when price is above EMA 200 (uptrend) but pulls back and crosses above EMA 50"""
    uptrend = df['close'] > df['ema_200']
    pullback_bounce = (df['close'].shift(1) < df['ema_50'].shift(1)) & (df['close'] > df['ema_50'])
    return uptrend & pullback_bounce

# --- NEW 15M INTRADAY CRYPTO STRATEGIES ---

def strat_supertrend_bullish(df):
    """Buy when Supertrend flips from Bearish (-1) to Bullish (+1)"""
    return (df['supertrend_dir'].shift(1) == -1) & (df['supertrend_dir'] == 1)

def strat_supertrend_bearish(df):
    """Short when Supertrend flips from Bullish (+1) to Bearish (-1)"""
    return (df['supertrend_dir'].shift(1) == 1) & (df['supertrend_dir'] == -1)

def strat_crypto_fib_stack(df):
    """Buy when fast crypto EMA 8 > EMA 21 > EMA 55 stack first forms"""
    aligned = (df['ema_8'] > df['ema_21']) & (df['ema_21'] > df['ema_55'])
    prev = (df['ema_8'].shift(1) > df['ema_21'].shift(1)) & (df['ema_21'].shift(1) > df['ema_55'].shift(1))
    return aligned & (~prev)

def strat_vwma_cross(df):
    """Buy when Volume-Weighted Moving Average (VWMA 8) crosses above VWMA 21"""
    return (df['vwma_8'].shift(1) <= df['vwma_21'].shift(1)) & (df['vwma_8'] > df['vwma_21'])

STRATEGIES = {
    'Supertrend Bullish (10,3)':   {'fn': strat_supertrend_bullish,   'direction': 'LONG'},
    'Supertrend Bearish (10,3)':   {'fn': strat_supertrend_bearish,   'direction': 'SHORT'},
    'Crypto Fib Stack (8/21/55)':  {'fn': strat_crypto_fib_stack,     'direction': 'LONG'},
    'VWMA Cross (8/21 Volume)':    {'fn': strat_vwma_cross,           'direction': 'LONG'},
    'StochRSI Oversold':           {'fn': strat_stoch_rsi_oversold,   'direction': 'LONG'},
    'StochRSI Overbought':         {'fn': strat_stoch_rsi_overbought, 'direction': 'SHORT'},
    'BB Squeeze Breakout':         {'fn': strat_bb_squeeze_breakout,  'direction': 'LONG'},
    'EMA 200/50 Pullback':         {'fn': strat_ema_pullback,         'direction': 'LONG'},
    'RSI Oversold Bounce':         {'fn': strat_rsi_oversold,          'direction': 'LONG'},
    'RSI Overbought Short':        {'fn': strat_rsi_overbought,        'direction': 'SHORT'},
    'Bullish MACD Cross':          {'fn': strat_macd_bull_cross,       'direction': 'LONG'},
    'Bearish MACD Cross':          {'fn': strat_macd_bear_cross,       'direction': 'SHORT'},
    'EMA 9/21 Golden Cross':       {'fn': strat_ema_golden_cross,      'direction': 'LONG'},
    'EMA 9/21 Death Cross':        {'fn': strat_ema_death_cross,       'direction': 'SHORT'},
    'BB Lower Bounce':             {'fn': strat_bb_lower_bounce,       'direction': 'LONG'},
    'BB Upper Reject':             {'fn': strat_bb_upper_reject,       'direction': 'SHORT'},
    'Volume Surge + RSI Dip':      {'fn': strat_volume_rsi_confirm,    'direction': 'LONG'},
    'Triple EMA Stack':            {'fn': strat_triple_ema_stack,      'direction': 'LONG'},
    'Mean Reversion':              {'fn': strat_mean_reversion,        'direction': 'LONG'},
    'Momentum Breakout':           {'fn': strat_momentum_breakout,     'direction': 'LONG'},
}

print(f'{len(STRATEGIES)} strategies defined (including 15m intraday Supertrend & VWMA).')


Defined 12 strategies for backtesting.


## 3. Backtesting Engine
The engine runs each strategy across all pairs, measures forward returns for each hold period, and computes performance metrics.

In [4]:
def backtest_strategy(df, signal_mask, direction, hold_period):
    """
    Given a DataFrame, boolean signal mask, trade direction, and hold period,
    compute per-trade returns and aggregate performance metrics.
    """
    df = df.copy()
    df['future_close'] = df['close'].shift(-hold_period)
    
    if direction == 'LONG':
        df['trade_return'] = ((df['future_close'] - df['close']) / df['close']) * 100
    else:  # SHORT
        df['trade_return'] = ((df['close'] - df['future_close']) / df['close']) * 100
    
    trades = df[signal_mask].dropna(subset=['trade_return']).copy()
    
    if len(trades) == 0:
        return None
    
    returns = trades['trade_return']
    wins = returns[returns > 0]
    losses = returns[returns <= 0]
    
    win_rate = (len(wins) / len(returns)) * 100
    avg_return = returns.mean()
    median_return = returns.median()
    total_return = ((1 + returns / 100).prod() - 1) * 100
    
    # Max drawdown from cumulative equity curve
    cum_returns = (1 + returns / 100).cumprod()
    rolling_max = cum_returns.cummax()
    drawdown = ((cum_returns - rolling_max) / rolling_max) * 100
    max_drawdown = drawdown.min()
    
    # Profit Factor
    gross_profit = wins.sum() if len(wins) > 0 else 0
    gross_loss = abs(losses.sum()) if len(losses) > 0 else 1e-10
    profit_factor = gross_profit / gross_loss
    
    # Sharpe Ratio (annualized, assuming hourly candles)
    if returns.std() > 0:
        sharpe = (returns.mean() / returns.std()) * np.sqrt(8760 / hold_period)
    else:
        sharpe = 0.0
    
    return {
        'total_trades': len(returns),
        'wins': len(wins),
        'losses': len(losses),
        'win_rate': round(win_rate, 2),
        'avg_return': round(avg_return, 4),
        'median_return': round(median_return, 4),
        'total_return': round(total_return, 4),
        'max_drawdown': round(max_drawdown, 4),
        'profit_factor': round(profit_factor, 4),
        'sharpe_ratio': round(sharpe, 4),
        'best_trade': round(returns.max(), 4),
        'worst_trade': round(returns.min(), 4),
    }

print('Backtesting engine ready.')

Backtesting engine ready.


## 4. Run All Strategies Across All Pairs

In [5]:
all_results = []

for strat_name, strat_info in STRATEGIES.items():
    fn = strat_info['fn']
    direction = strat_info['direction']
    
    for hold in HOLD_PERIODS:
        # Aggregate signals across all symbols
        agg_trades = []
        total_signals = 0
        
        for sym, df in datasets.items():
            try:
                signals = fn(df)
                n_signals = signals.sum()
                total_signals += n_signals
                
                if n_signals > 0:
                    result = backtest_strategy(df, signals, direction, hold)
                    if result:
                        # Store per-symbol for aggregation
                        df_temp = df.copy()
                        df_temp['future_close'] = df_temp['close'].shift(-hold)
                        if direction == 'LONG':
                            df_temp['trade_return'] = ((df_temp['future_close'] - df_temp['close']) / df_temp['close']) * 100
                        else:
                            df_temp['trade_return'] = ((df_temp['close'] - df_temp['future_close']) / df_temp['close']) * 100
                        trade_returns = df_temp[signals].dropna(subset=['trade_return'])['trade_return']
                        agg_trades.extend(trade_returns.tolist())
            except Exception:
                continue
        
        # Compute aggregate metrics
        if len(agg_trades) > 0:
            returns = pd.Series(agg_trades)
            wins = returns[returns > 0]
            losses = returns[returns <= 0]
            
            cum_returns = (1 + returns / 100).cumprod()
            rolling_max = cum_returns.cummax()
            drawdown = ((cum_returns - rolling_max) / rolling_max) * 100
            
            gross_profit = wins.sum() if len(wins) > 0 else 0
            gross_loss = abs(losses.sum()) if len(losses) > 0 else 1e-10
            
            sharpe = (returns.mean() / returns.std()) * np.sqrt(8760 / hold) if returns.std() > 0 else 0
            
            all_results.append({
                'Strategy': strat_name,
                'Direction': direction,
                'Hold (hrs)': hold,
                'Signals': len(returns),
                'Win Rate (%)': round((len(wins) / len(returns)) * 100, 2),
                'Avg Return (%)': round(returns.mean(), 4),
                'Median Return (%)': round(returns.median(), 4),
                'Total Return (%)': round(((1 + returns / 100).prod() - 1) * 100, 4),
                'Max Drawdown (%)': round(drawdown.min(), 4),
                'Profit Factor': round(gross_profit / gross_loss, 4),
                'Sharpe Ratio': round(sharpe, 4),
                'Best Trade (%)': round(returns.max(), 4),
                'Worst Trade (%)': round(returns.min(), 4),
            })

df_results = pd.DataFrame(all_results)
print(f'Backtesting complete: {len(df_results)} strategy-holdperiod combinations evaluated.\n')
df_results.sort_values('Sharpe Ratio', ascending=False).head(10)

Backtesting complete: 36 strategy-holdperiod combinations evaluated.



,Strategy,Direction,Hold (hrs),Signals,Win Rate (%),Avg Return (%),Median Return (%),Total Return (%),Max Drawdown (%),Profit Factor,Sharpe Ratio,Best Trade (%),Worst Trade (%)
24,Volume Surge + RSI Dip,LONG,6,111,63.06,0.5940,0.4333,91.2402,-7.3976,3.4497,17.5629,3.1898,-3.7092
25,Volume Surge + RSI Dip,LONG,12,111,65.77,0.6383,0.4613,100.1652,-6.3507,3.2255,11.4412,4.8765,-4.1729
18,BB Lower Band Bounce,LONG,6,93,61.29,0.2546,0.2954,25.6034,-9.7674,1.6325,7.1433,3.1840,-4.3959
0,RSI Oversold Bounce,LONG,6,102,55.88,0.2438,0.0967,26.6926,-8.8198,1.7607,5.9782,11.1427,-3.1505
30,Mean Reversion,LONG,6,106,55.66,0.2026,0.1819,22.6649,-14.9791,1.4722,5.5276,3.1898,-3.1505
31,Mean Reversion,LONG,12,106,48.11,0.3219,0.0000,38.6726,-15.6449,1.6979,5.3552,4.3616,-2.9308
19,BB Lower Band Bounce,LONG,12,93,53.76,0.3378,0.0928,34.8459,-9.9517,1.6357,5.0960,4.3616,-4.7099
1,RSI Oversold Bounce,LONG,12,102,52.94,0.2635,0.0977,28.8863,-14.1727,1.5636,4.1365,9.3235,-5.6818
8,Bullish MACD Cross,LONG,24,154,62.34,0.5448,0.6278,120.0209,-15.7897,1.7340,4.1241,7.3583,-6.5278
14,EMA 9/21 Golden Cross,LONG,24,94,64.89,0.5147,0.7199,57.6795,-11.6656,1.7407,4.0533,7.1061,-6.5278


## 5. Strategy Comparison — Full Ranked Table (6h Hold)

In [6]:
# Focus on 6-hour hold period for main comparison
df_6h = df_results[df_results['Hold (hrs)'] == 6].copy()
df_6h = df_6h.sort_values('Sharpe Ratio', ascending=False).reset_index(drop=True)
df_6h.index = df_6h.index + 1
df_6h.index.name = 'Rank'

print('='*100)
print('STRATEGY PERFORMANCE RANKING — 6 Hour Hold Period')
print('='*100)
df_6h[['Strategy', 'Direction', 'Signals', 'Win Rate (%)', 'Avg Return (%)', 
        'Total Return (%)', 'Max Drawdown (%)', 'Profit Factor', 'Sharpe Ratio']]

STRATEGY PERFORMANCE RANKING — 6 Hour Hold Period


,Strategy,Direction,Signals,Win Rate (%),Avg Return (%),Total Return (%),Max Drawdown (%),Profit Factor,Sharpe Ratio
Rank,,,,,,,,,
1,Volume Surge + RSI Dip,LONG,111,63.06,0.5940,91.2402,-7.3976,3.4497,17.5629
2,BB Lower Band Bounce,LONG,93,61.29,0.2546,25.6034,-9.7674,1.6325,7.1433
3,RSI Oversold Bounce,LONG,102,55.88,0.2438,26.6926,-8.8198,1.7607,5.9782
4,Mean Reversion,LONG,106,55.66,0.2026,22.6649,-14.9791,1.4722,5.5276
5,Bullish MACD Cross,LONG,156,56.41,0.0892,13.9548,-6.2335,1.2599,3.2503
6,BB Upper Band Reject,SHORT,129,53.49,0.0788,9.8575,-13.2138,1.2178,2.7671
7,EMA 9/21 Golden Cross,LONG,95,49.47,0.0725,6.4271,-6.8938,1.1833,2.3417
8,Triple EMA Stack,LONG,76,52.63,-0.0216,-2.1048,-11.7609,0.9497,-0.7200
9,RSI Overbought Short,SHORT,141,53.19,-0.0498,-7.7698,-21.6724,0.8904,-1.5531


## 6. Visual Comparison Charts

In [7]:
# Win Rate vs Avg Return scatter plot (6h hold)
fig = px.scatter(
    df_6h,
    x='Win Rate (%)',
    y='Avg Return (%)',
    size='Signals',
    color='Sharpe Ratio',
    color_continuous_scale='RdYlGn',
    hover_name='Strategy',
    text='Strategy',
    title='Strategy Win Rate vs Average Return (6h Hold)',
    labels={'Signals': 'Number of Signals'},
    template='plotly_dark'
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=50, line_dash='dash', line_color='gray', opacity=0.5)
fig.update_layout(height=600)
fig.show()

In [8]:
# Bar chart comparing key metrics across strategies
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Win Rate (%)', 'Avg Return per Trade (%)', 'Profit Factor', 'Sharpe Ratio'),
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

strategies = df_6h['Strategy'].tolist()
colors = px.colors.qualitative.Vivid[:len(strategies)]

fig.add_trace(go.Bar(x=strategies, y=df_6h['Win Rate (%)'].tolist(), marker_color=colors, showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=strategies, y=df_6h['Avg Return (%)'].tolist(), marker_color=colors, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=strategies, y=df_6h['Profit Factor'].tolist(), marker_color=colors, showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=strategies, y=df_6h['Sharpe Ratio'].tolist(), marker_color=colors, showlegend=False), row=2, col=2)

fig.update_layout(height=700, template='plotly_dark', title_text='Strategy Comparison Dashboard (6h Hold)')
fig.update_xaxes(tickangle=45, tickfont_size=8)
fig.show()

## 7. Performance Across Hold Periods — Which Timeframe Works Best?

In [9]:
# Heatmap of Win Rate across strategies and hold periods
pivot_wr = df_results.pivot_table(index='Strategy', columns='Hold (hrs)', values='Win Rate (%)', aggfunc='first')
pivot_wr = pivot_wr.reindex(df_6h['Strategy'].tolist())  # Sort by 6h ranking

fig = go.Figure(data=go.Heatmap(
    z=pivot_wr.values,
    x=[f'{h}h Hold' for h in pivot_wr.columns],
    y=pivot_wr.index,
    colorscale='RdYlGn',
    text=np.round(pivot_wr.values, 1),
    texttemplate='%{text}%',
    textfont={'size': 11},
    colorbar_title='Win Rate (%)'
))
fig.update_layout(
    title='Win Rate Heatmap: Strategy × Hold Period',
    height=500,
    template='plotly_dark',
    yaxis=dict(autorange='reversed')
)
fig.show()

In [10]:
# Heatmap of Sharpe Ratio across strategies and hold periods
pivot_sharpe = df_results.pivot_table(index='Strategy', columns='Hold (hrs)', values='Sharpe Ratio', aggfunc='first')
pivot_sharpe = pivot_sharpe.reindex(df_6h['Strategy'].tolist())

fig = go.Figure(data=go.Heatmap(
    z=pivot_sharpe.values,
    x=[f'{h}h Hold' for h in pivot_sharpe.columns],
    y=pivot_sharpe.index,
    colorscale='RdYlGn',
    text=np.round(pivot_sharpe.values, 2),
    texttemplate='%{text}',
    textfont={'size': 11},
    colorbar_title='Sharpe Ratio'
))
fig.update_layout(
    title='Sharpe Ratio Heatmap: Strategy × Hold Period',
    height=500,
    template='plotly_dark',
    yaxis=dict(autorange='reversed')
)
fig.show()

## 8. Deep Dive: Top 3 Strategies — Equity Curves

In [11]:
# Plot cumulative equity curve for top 3 strategies on BTC (6h hold)
top_3 = df_6h.head(3)['Strategy'].tolist()
hold = 6
btc_df = datasets.get('BTCUSDT')

if btc_df is not None:
    fig = go.Figure()
    
    for strat_name in top_3:
        fn = STRATEGIES[strat_name]['fn']
        direction = STRATEGIES[strat_name]['direction']
        signals = fn(btc_df)
        
        df_temp = btc_df.copy()
        df_temp['future_close'] = df_temp['close'].shift(-hold)
        if direction == 'LONG':
            df_temp['trade_return'] = ((df_temp['future_close'] - df_temp['close']) / df_temp['close']) * 100
        else:
            df_temp['trade_return'] = ((df_temp['close'] - df_temp['future_close']) / df_temp['close']) * 100
        
        trade_data = df_temp[signals].dropna(subset=['trade_return'])
        
        if len(trade_data) > 0:
            equity = (1 + trade_data['trade_return'] / 100).cumprod() * 100
            fig.add_trace(go.Scatter(
                x=trade_data['open_time'],
                y=equity.values,
                mode='lines+markers',
                name=f'{strat_name} ({len(trade_data)} trades)',
                line=dict(width=2)
            ))
    
    fig.add_hline(y=100, line_dash='dash', line_color='white', opacity=0.3, annotation_text='Starting Capital')
    fig.update_layout(
        title='Top 3 Strategies — Cumulative Equity Curve (BTCUSDT, 6h Hold)',
        yaxis_title='Portfolio Value (Starting: $100)',
        xaxis_title='Date',
        height=500,
        template='plotly_dark'
    )
    fig.show()
else:
    print('BTCUSDT data not available for equity curve.')

## 9. Per-Symbol Breakdown for the Best Strategy

In [12]:
best_strat_name = df_6h.iloc[0]['Strategy']
best_strat = STRATEGIES[best_strat_name]
hold = 6

print(f'Best Strategy: {best_strat_name} ({best_strat["direction"]})')
print('='*80)

per_symbol_results = []

for sym, df in datasets.items():
    signals = best_strat['fn'](df)
    result = backtest_strategy(df, signals, best_strat['direction'], hold)
    if result:
        result['Symbol'] = sym
        per_symbol_results.append(result)

df_per_sym = pd.DataFrame(per_symbol_results)
if not df_per_sym.empty:
    df_per_sym = df_per_sym[['Symbol', 'total_trades', 'win_rate', 'avg_return', 'total_return', 
                              'max_drawdown', 'profit_factor', 'sharpe_ratio', 'best_trade', 'worst_trade']]
    df_per_sym.columns = ['Symbol', 'Trades', 'Win Rate (%)', 'Avg Return (%)', 'Total Return (%)',
                          'Max DD (%)', 'Profit Factor', 'Sharpe', 'Best Trade (%)', 'Worst Trade (%)']
    df_per_sym = df_per_sym.sort_values('Sharpe', ascending=False).reset_index(drop=True)
    display(df_per_sym)
else:
    print('No signals generated for any symbol.')

Best Strategy: Volume Surge + RSI Dip (LONG)


,Symbol,Trades,Win Rate (%),Avg Return (%),Total Return (%),Max DD (%),Profit Factor,Sharpe,Best Trade (%),Worst Trade (%)
0,DOGEUSDT,17,82.35,0.9853,18.0450,-1.2746,13.3140,37.8143,2.6449,-0.8485
1,BNBUSDT,11,72.73,0.7379,8.3845,-0.7769,11.4308,33.1445,2.1768,-0.5319
2,SOLUSDT,9,66.67,0.8679,8.0251,-0.6362,12.1203,27.1255,3.1840,-0.4558
3,ETHUSDT,12,66.67,0.6792,8.3885,-0.6259,9.3165,23.0612,2.9773,-0.3285
4,BTCUSDT,21,66.67,0.5226,11.4290,-2.6654,4.0742,17.7424,3.1898,-1.0598
5,ADAUSDT,11,63.64,0.5754,6.3697,-4.6416,2.3415,13.2592,3.1697,-1.8519
6,XRPUSDT,14,42.86,0.4516,6.3643,-2.9395,2.4198,11.7809,2.9570,-1.8446
7,AVAXUSDT,16,43.75,0.0928,1.2615,-7.3976,1.1434,2.0240,3.1435,-3.7092


## 10. Full Results Export & Final Verdict

In [13]:
# Final summary sorted by composite score
df_final = df_6h.copy()

# Composite score: normalized win_rate + profit_factor + sharpe
if len(df_final) > 1:
    for col in ['Win Rate (%)', 'Profit Factor', 'Sharpe Ratio']:
        col_min = df_final[col].min()
        col_max = df_final[col].max()
        rng = col_max - col_min if col_max != col_min else 1
        df_final[f'{col}_norm'] = (df_final[col] - col_min) / rng
    
    df_final['Composite Score'] = (
        df_final['Win Rate (%)_norm'] * 0.3 + 
        df_final['Profit Factor_norm'] * 0.35 + 
        df_final['Sharpe Ratio_norm'] * 0.35
    ).round(4)
    df_final = df_final.sort_values('Composite Score', ascending=False)

print('='*100)
print('FINAL STRATEGY RANKING BY COMPOSITE SCORE (6h Hold)')
print('Win Rate (30%) + Profit Factor (35%) + Sharpe Ratio (35%)')
print('='*100)

display_cols = ['Strategy', 'Direction', 'Signals', 'Win Rate (%)', 'Avg Return (%)',
                'Total Return (%)', 'Max Drawdown (%)', 'Profit Factor', 'Sharpe Ratio']
if 'Composite Score' in df_final.columns:
    display_cols.append('Composite Score')

df_display = df_final[display_cols].reset_index(drop=True)
df_display.index = df_display.index + 1
df_display.index.name = 'Rank'
display(df_display)

# Print the winner
best = df_display.iloc[0]
print(f'\n{"="*60}')
print(f'  BEST STRATEGY: {best["Strategy"]}')
print(f'  Direction: {best["Direction"]}  |  Win Rate: {best["Win Rate (%)"]:.1f}%  |  Avg Return: {best["Avg Return (%)"]:.4f}%')
print(f'  Profit Factor: {best["Profit Factor"]:.2f}  |  Sharpe Ratio: {best["Sharpe Ratio"]:.2f}')
print(f'{"="*60}')

FINAL STRATEGY RANKING BY COMPOSITE SCORE (6h Hold)
Win Rate (30%) + Profit Factor (35%) + Sharpe Ratio (35%)


,Strategy,Direction,Signals,Win Rate (%),Avg Return (%),Total Return (%),Max Drawdown (%),Profit Factor,Sharpe Ratio,Composite Score
Rank,,,,,,,,,,
1,Volume Surge + RSI Dip,LONG,111,63.06,0.5940,91.2402,-7.3976,3.4497,17.5629,1.0000
2,BB Lower Band Bounce,LONG,93,61.29,0.2546,25.6034,-9.7674,1.6325,7.1433,0.5877
3,RSI Oversold Bounce,LONG,102,55.88,0.2438,26.6926,-8.8198,1.7607,5.9782,0.5145
4,Mean Reversion,LONG,106,55.66,0.2026,22.6649,-14.9791,1.4722,5.5276,0.4681
5,Bullish MACD Cross,LONG,156,56.41,0.0892,13.9548,-6.2335,1.2599,3.2503,0.4165
6,BB Upper Band Reject,SHORT,129,53.49,0.0788,9.8575,-13.2138,1.2178,2.7671,0.3651
7,EMA 9/21 Golden Cross,LONG,95,49.47,0.0725,6.4271,-6.8938,1.1833,2.3417,0.3010
8,Triple EMA Stack,LONG,76,52.63,-0.0216,-2.1048,-11.7609,0.9497,-0.7200,0.2667
9,RSI Overbought Short,SHORT,141,53.19,-0.0498,-7.7698,-21.6724,0.8904,-1.5531,0.2539



  BEST STRATEGY: Volume Surge + RSI Dip
  Direction: LONG  |  Win Rate: 63.1%  |  Avg Return: 0.5940%
  Profit Factor: 3.45  |  Sharpe Ratio: 17.56
